# step2_content_based

**Owner:** P2  
**Purpose:** Content-based filtering with title/abstract text similarity to a user's observed click history. This model can score cold articles because it uses article content, not an article-ID history.  
**Inputs:** `data/processed/news_metadata.parquet`, `eval_candidates.parquet`, and `eval_impressions.parquet` from Step 1  
**Outputs:** `outputs/results_step2_content_based.csv`, `data/scores/content_based.parquet` (one score for every 15 Nov evaluation candidate)

> Runtime: Google Colab. Run cells top to bottom; mount the shared Drive shortcut `BT4222_Project` when prompted.

In [ ]:
# --- Google Colab setup: run this first, every session ---
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path("/content/BT4222-Project-Group-4--MIND")
if not (REPO_ROOT / "config.py").is_file():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git",
            str(REPO_ROOT),
        ],
        check=True,
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

import config
from utils.metrics import add_rank, evaluate, evaluate_by_segment, topk, topk_diversity

PROC = Path(config.PROCESSED_DIR)
OUT = Path(config.OUTPUT_DIR)
SCORES = Path(config.SCORES_DIR)
STEP1_FILES = [
    PROC / "news_metadata.parquet",
    PROC / "eval_candidates.parquet",
    PROC / "eval_impressions.parquet",
]
missing = [str(file) for file in STEP1_FILES if not file.is_file()]
if missing:
    raise FileNotFoundError(
        "Run step1_data_preprocess.ipynb first and make sure its outputs are in "
        f"{PROC}. Missing files:\n  - " + "\n  - ".join(missing)
    )

print("Project code:", REPO_ROOT)
print("Google Drive project:", config.PROJECT_DIR)
print("Processed data:", PROC)

## 1. Build article text embeddings and user profiles

Represent each article by its title and abstract. TF/IDF followed by truncated SVD gives a compact, L2-normalised text vector; no click labels are used to learn these vectors. For each impression, the user profile is the mean of the vectors for articles in that impression's observed history, then L2-normalised. Articles missing from metadata are ignored in histories and reported.

In [ ]:
news = pd.read_parquet(
    PROC / "news_metadata.parquet",
    columns=["news_id", "title", "abstract"],
).drop_duplicates("news_id", keep="first").reset_index(drop=True)

article_text = (
    news["title"].fillna("").astype(str)
    + " "
    + news["abstract"].fillna("").astype(str)
).str.strip()
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=50_000,
    sublinear_tf=True,
)
tfidf = vectorizer.fit_transform(article_text)
if min(tfidf.shape) < 3:
    raise ValueError(f"Not enough article text for SVD embeddings: matrix shape {tfidf.shape}")

embedding_dim = min(128, tfidf.shape[0] - 1, tfidf.shape[1] - 1)
svd = TruncatedSVD(n_components=embedding_dim, n_iter=5, random_state=config.SEED)
article_embeddings = svd.fit_transform(tfidf).astype(np.float32)
article_norms = np.linalg.norm(article_embeddings, axis=1, keepdims=True)
article_embeddings /= np.maximum(article_norms, 1e-12)
article_to_row = dict(zip(news["news_id"], range(len(news))))

# Keep the Step 1 history field at impression level; it is time-specific.
eval_impressions = pd.read_parquet(
    PROC / "eval_impressions.parquet", columns=["impression_id", "history"]
)
if eval_impressions["impression_id"].duplicated().any():
    raise ValueError("eval_impressions.parquet contains duplicate impression_id values")

explained_variance = float(svd.explained_variance_ratio_.sum())
if not np.isfinite(explained_variance):
    raise ValueError("Article text has no measurable variance; check title/abstract metadata")

print(f"Articles: {len(news):,}; text features: {tfidf.shape[1]:,}; embedding dimensions: {embedding_dim}")
print(f"Explained variance captured by SVD: {explained_variance:.3f}")

## 2. Score each full evaluation slate with cosine similarity

For every 15 Nov impression, score each displayed article by cosine similarity between its text vector and the user's history profile. Process impressions in batches to bound temporary memory. A zero-history or all-unknown-history profile receives neutral zero scores; the shared metrics then break those ties reproducibly rather than inventing a preference.

In [ ]:
# Load only the columns needed for ranking and shared evaluation.
eval_candidates = pd.read_parquet(
    PROC / "eval_candidates.parquet",
    columns=["impression_id", "news_id", "label", "cold_article", "category"],
)
required = {"impression_id", "news_id", "label", "cold_article", "category"}
if not required.issubset(eval_candidates.columns):
    raise ValueError(f"Evaluation candidates are missing columns: {sorted(required - set(eval_candidates.columns))}")

impression_ids = pd.Index(eval_impressions["impression_id"])
candidate_impression_index = impression_ids.get_indexer(eval_candidates["impression_id"])
if (candidate_impression_index < 0).any():
    raise ValueError("Some evaluation candidates have no matching impression history")
article_ids = pd.Index(news["news_id"])
candidate_article_index = article_ids.get_indexer(eval_candidates["news_id"])
if (candidate_article_index < 0).any():
    missing_ids = eval_candidates.loc[candidate_article_index < 0, "news_id"].drop_duplicates().head(10).tolist()
    raise ValueError(f"Candidate articles are missing from news_metadata.parquet, e.g. {missing_ids}")

history_values = eval_impressions["history"].fillna("").astype(str).to_numpy()
candidate_counts = np.bincount(candidate_impression_index, minlength=len(eval_impressions))
candidate_order = np.argsort(candidate_impression_index, kind="stable")
candidate_offsets = np.concatenate(([0], np.cumsum(candidate_counts, dtype=np.int64)))
candidate_scores = np.zeros(len(eval_candidates), dtype=np.float32)
has_history = np.zeros(len(eval_impressions), dtype=bool)
history_article_count = np.zeros(len(eval_impressions), dtype=np.int32)
missing_history_articles = 0

impressions_per_batch = 2_000
for batch_start in range(0, len(eval_impressions), impressions_per_batch):
    batch_end = min(batch_start + impressions_per_batch, len(eval_impressions))
    profiles = np.zeros((batch_end - batch_start, embedding_dim), dtype=np.float32)

    for local_index, history in enumerate(history_values[batch_start:batch_end]):
        history_ids = history.split()
        rows = [article_to_row[news_id] for news_id in history_ids if news_id in article_to_row]
        missing_history_articles += len(history_ids) - len(rows)
        history_article_count[batch_start + local_index] = len(rows)
        if not rows:
            continue
        profile = article_embeddings[rows].mean(axis=0)
        profile_norm = np.linalg.norm(profile)
        if profile_norm > 0:
            profiles[local_index] = profile / profile_norm
            has_history[batch_start + local_index] = True

    sorted_start = int(candidate_offsets[batch_start])
    sorted_end = int(candidate_offsets[batch_end])
    ordered_rows = candidate_order[sorted_start:sorted_end]
    local_impression_index = candidate_impression_index[ordered_rows] - batch_start
    candidate_vectors = article_embeddings[candidate_article_index[ordered_rows]]
    candidate_scores[ordered_rows] = np.einsum(
        "ij,ij->i", candidate_vectors, profiles[local_impression_index]
    )

labels = eval_candidates["label"].to_numpy()
if not np.isin(labels, [0, 1]).all():
    raise ValueError("Evaluation labels must be binary; use the full labelled Step 1 dev split")
if labels.mean() >= 0.10:
    raise ValueError(
        f"Evaluation click rate {labels.mean():.3f} is too high for full slates; "
        "do not evaluate on the negative-sampled fit split."
    )

eval_candidates = eval_candidates.rename(columns={"cold_article": "is_cold"}).copy()
eval_candidates["is_cold"] = eval_candidates["is_cold"].astype(np.int8)
eval_candidates["score"] = candidate_scores
eval_candidates["has_history"] = has_history[candidate_impression_index]

print(f"Evaluation candidates: {len(eval_candidates):,}; impressions: {len(eval_impressions):,}")
print(f"Impressions with usable history: {has_history.sum():,} ({has_history.mean():.1%})")
print(f"History article IDs absent from news metadata: {missing_history_articles:,}")

## 3. Users with empty or unavailable history

The model does not fabricate a user preference. If an impression has no history articles represented in the metadata, every candidate receives score 0, making that impression a neutral tie slate. We report how many impressions have a usable profile and later show metrics for those slates separately from empty-history slates.

In [ ]:
history_diagnostics = pd.Series(
    {
        "evaluation impressions": len(eval_impressions),
        "usable history impressions": int(has_history.sum()),
        "empty or unavailable history impressions": int((~has_history).sum()),
        "mean known history articles": round(float(history_article_count.mean()), 2),
        "history articles absent from metadata": int(missing_history_articles),
    },
    name="count / mean",
)
display(history_diagnostics.to_frame())

## 4. Results: overall, cold vs warm, and history coverage

Use the shared project metrics on complete 15 Nov slates. The cold/warm segment is defined by the clicked article's Step 1 `cold_article` flag. Also report top-10 category diversity and cold share, and separate overall relevance for impressions with usable versus empty histories.

In [ ]:
results = evaluate_by_segment(eval_candidates, "score", "is_cold")
top10 = topk(add_rank(eval_candidates, "score"), "rank", k=config.TOP_K)
top10_summary = topk_diversity(
    top10,
    category_col="category",
    cold_col="is_cold",
    k=config.TOP_K,
)
for metric, value in top10_summary.items():
    results.loc["all", f"top{config.TOP_K}_{metric}"] = value

impression_has_history = pd.Series(has_history, index=impression_ids)
for segment, has_usable_history in (("has_history", True), ("empty_history", False)):
    segment_ids = impression_has_history.index[impression_has_history == has_usable_history]
    if len(segment_ids):
        segment_candidates = eval_candidates[eval_candidates["impression_id"].isin(segment_ids)]
        results.loc[segment] = evaluate(segment_candidates, "score")

OUT.mkdir(parents=True, exist_ok=True)
SCORES.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT / "results_step2_content_based.csv", index=True, index_label="segment")
score_file = SCORES / "content_based.parquet"
score_output = eval_candidates[["impression_id", "news_id", "score"]]
if score_output.duplicated(["impression_id", "news_id"]).any():
    raise ValueError("Duplicate impression/article pairs found; refusing to write an ambiguous score file")
if score_output["score"].isna().any():
    raise ValueError("NaN scores found; refusing to write an incomplete score file")
score_output.to_parquet(score_file, index=False)

display(results.round(4))
print("Results saved:", OUT / "results_step2_content_based.csv")
print("All-candidate score handoff saved:", score_file)

## 5. Past group (Group 10): contextual benchmark

Group 10's content-based notebook reports its test results on MIND-small. Put those reported values next to our MIND-large full-slate results, but interpret the comparison as context rather than a controlled benchmark: dataset size, split construction, model features, and metric definitions differ. In particular, Group 10's MRR uses only the first clicked item's reciprocal rank, while our shared `utils/metrics.py` averages reciprocal ranks over all clicked articles. The prior test did not report cold/warm segments.

In [ ]:
group10_test = pd.Series(
    {
        "auc": 0.5782,
        "mrr": 0.2825,
        "ndcg@5": 0.2836,
        "ndcg@10": 0.3373,
    },
    name="Group 10 reported test (MIND-small)",
)
our_test = results.loc["all", ["auc", "mrr", "ndcg@5", "ndcg@10"]].astype(float)
comparison = pd.concat([group10_test, our_test.rename("This project (MIND-large, 15 Nov full slates)")], axis=1)
display(comparison.round(4))

print(
    "Group 10 source: https://github.com/bt422210/BT4222-Group-10/blob/main/4_Basic_Models/Content_Based_Filtering_Model.ipynb\n"
    "Their README identifies MIND-small; the results are from their own test split and feature/model pipeline. "
    "Do not interpret the differences as a model-only effect."
)